<a href="https://colab.research.google.com/github/Apur52027/depression_detection/blob/main/tf_df_lr_lgbm_xgb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize

# Load the dataset
df = pd.read_excel('https://github.com/Apur52027/Dataset_ML/blob/main/BSMDD_v3_textcleaned%20-%2021K.xlsx?raw=true')

# 1. Remove Duplicate Texts
initial_count = len(df)
df = df.drop_duplicates(subset=['text'])
print(f"Removed {initial_count - len(df)} duplicate texts.")

# 2. Remove Short Texts (less than 30 words)
# Word count for Bengali is tricky. We'll split by spaces as a proxy.
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 30].drop(columns=['word_count'])
print(f"Removed texts with less than 30 words. Remaining rows: {len(df)}")

Removed 6 duplicate texts.
Removed texts with less than 30 words. Remaining rows: 19178


In [ ]:
# 3. Remove English characters and numbers
# This regex keeps Bengali unicode range (\u0980-\u09FF), whitespace, and a few punctuations.
def remove_english_and_numbers(text):
    # Keep Bengali chars, spaces, and basic punctuation (। ? !)
    bengali_pattern = re.compile(r'[^\u0980-\u09FF\s\?\!।]', re.UNICODE)
    return re.sub(bengali_pattern, '', str(text))

df['cleaned_text'] = df['text'].apply(remove_english_and_numbers)

# 4. Remove repeated punctuations
def remove_repeated_punct(text):
    # Replace 2 or more '?' or '!' or '।' with a single instance
    text = re.sub(r'([\?\!।])\1+', r'\1', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['cleaned_text'].apply(remove_repeated_punct)

# At this point, you have a 'cleaned_text' column ready for tokenization.

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
import unicodedata # Added this import
# Download stopwords if not already
nltk.download('stopwords')

bangla_stopwords = set(stopwords.words('bengali'))

def preprocess_text(text):
    # Tokenize
     # Tokenization
    tokens = text.split()
    # remove stopword
    tokens = [word for word in tokens if word not in bangla_stopwords]
    process = " ".join(tokens)
    return process # Added return statement


df['final_text'] = df['cleaned_text'].apply(preprocess_text)

# Final dataset
X = df['final_text']  # Features
y = df['label']       # Labels (1 for depressed, 0 for non-depressed)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
import re

def check_unwanted_characters(text):
    # This pattern finds any character that is NOT a Bengali character (\u0980-\u09FF),
    # a space (\s), or one of the allowed punctuations (\?|!|\u0964 - for Bengali full stop).
    # It also explicitly excludes English letters and numbers, which should already be gone.
    unwanted_pattern = re.compile(r'[^\u0980-\u09FF\s\?!\u0964]', re.UNICODE)
    found_unwanted = set(unwanted_pattern.findall(str(text)))
    return found_unwanted

all_unwanted_chars = set()
for text in df['final_text']:
    unwanted = check_unwanted_characters(text)
    if unwanted:
        all_unwanted_chars.update(unwanted)

if all_unwanted_chars:
    print(f"Found unwanted characters in 'final_text': {', '.join(sorted(list(all_unwanted_chars)))}")
else:
    print("No unwanted characters (non-Bengali, non-punctuation, non-space) found in 'final_text'.")

# Optional: Display some rows where unwanted characters might have been found (if any)
# For demonstration, let's just show a few rows of final_text to confirm its content
print("\nSample of 'final_text' column:")
display(df['final_text'].head())

No unwanted characters (non-Bengali, non-punctuation, non-space) found in 'final_text'.

Sample of 'final_text' column:


,final_text
0,মানসিক শারীরিকভাবে অসুস্থ ক্লান্ত পুরো জীবন শা...
1,দয়া সাথে থাকুন অত্যন্ত দীর্ঘ আপনাকে পড়তে উত্...
2,জানতাম সাথে ভুল লোক খারাপ জীবন কাটিয়েছে সম্পূ...
3,অনেটিভ ইংরেজি স্পিকারের অনুসরণ বিরক্তিকর অপ্রত...
4,অনেটিভ ইংরেজি স্পিকারের অনুসরণ বিরক্তিকর অপ্রত...


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.model_selection import train_test_split
X = df['final_text']
y = df['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
!pip install xgboost

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier # Corrected import
import lightgbm as lgb
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

# Common TF-IDF
tfidf = TfidfVectorizer(
    max_features=12000,
    min_df=3,
    max_df=0.85,
    ngram_range=(1, 2),
    # token_pattern=r'(?u)[\w]+\b',
    sublinear_tf=True
)

# Different models
models = {
    "lr": LogisticRegression(
    max_iter=200,
    C=1.0,
    penalty='l2',
    solver='lbfgs',
    random_state=42,
    class_weight="balanced"
),
    "rf": RandomForestClassifier(
    n_estimators=300,       # বেশি tree → better stability
    max_depth=None,         # default, but explicitly দিলে clear হয়
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
),
    "svm":SVC(
    kernel='rbf',        # most commonly used
    C=1.0,               # regularization
    gamma='scale',       # auto ভালো কাজ করে
    probability=True,    # probability দরকার হলে
    random_state=42
),
    "nb": MultinomialNB(
    alpha=0.5,   # smoothing (tune করা যায়: 0.1, 0.5, 1.0)
    fit_prior=True
),
    'xgb': XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
),
    'lgbm': lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
),
    'dt': DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    random_state=42
),
    'knn': KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    algorithm='auto',
    metric='minkowski',
    p=2
),
     'gd': GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
),
    'ada': AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)



}


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Base learners (level-0 models)
estimators = [
    # ('svm', models['svm']),
    ('lr', models['lr']),
    ('lgbm', models['lgbm']),
    ('xgb', models['xgb']),
    # ('nb', models['nb']),
    # ('rf', models['rf']),
    # ('gd', models['gd'])
]

# Meta learner (level-1 model)
meta_model = LogisticRegression(
    max_iter=200,
    C=1.0,
    class_weight='balanced',
    random_state=42
)

# Stacking model
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    cv=5,              # cross-validation
    n_jobs=-1,
    passthrough=False  # True করলে original features + predictions use করবে
)

# Full pipeline
stack_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('stack', stacking_model)
])

# Train
stack_pipeline.fit(X_train, y_train)

# Predict
y_pred = stack_pipeline.predict(X_test)

# Accuracy
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.8644421272158499
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      2163
           1       0.83      0.86      0.85      1673

    accuracy                           0.86      3836
   macro avg       0.86      0.86      0.86      3836
weighted avg       0.87      0.86      0.86      3836

